In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.axes_grid1 import make_axes_locatable
import matplotlib as mpl
from matplotlib.ticker import AutoLocator, AutoMinorLocator, LogLocator
import glob
from scipy.interpolate import griddata
from pathlib import Path
import h5py
import sys
from pathlib import Path
import os

# Where am I running?
try:
    # Normal script
    here = Path(__file__).resolve().parent
except NameError:
    # Notebook / REPL
    here = Path.cwd()

phys_const_path = (here / '..' / 'phys_const').resolve()
sys.path.append(str(phys_const_path))

nsm_plots_path = (here / '..' / 'nsm_plots').resolve()
sys.path.append(str(nsm_plots_path))

nsm_plots_postproc = (here / '..' / 'nsm_instabilities').resolve()
sys.path.append(str(nsm_plots_postproc))

import phys_const as pc
import plot_functions as pf

### Studying the trend of quantum coherence in a global post-merger accretion disk as a function of the attenuation factor. Simulation with 92 PPEBPC and 1 cubic-kilometer cells.

In [ ]:
# # HAR
har_rho_eu_48_48=8.237993097357763e-06
har_rho_eu_32_16=1.6913100570908646e-09
# np.average(rho_eu[5:40,16])=1.8255508397170877e-09

# # HSR
hsr_rho_eu_96_60=6.839131272287458e-06
hsr_rho_eu_64_32=2.3495318539521587e-07
# np.average(rho_eu[5:80,32])=2.4813035321102116e-07

In [ ]:
eta = np.array([1e-5,1e-6,1e-7,1e-8,1e-9,1e-10,1e-11,1e-12])
rho_eu_48_48 = np.array([8.28936886418095e-06, 2.8740007172122207e-07, 2.3136689954208057e-08, 2.81132878168119e-09, 2.8437389366196645e-10, 2.8440521140415826e-11, 2.844055244298335e-12, 2.8440552756007335e-13])

log_eta = np.log10(eta)
log_rho_eu_48_48 = np.log10(rho_eu_48_48)

# Fit a straight line: log_frequency = slope * log_eta + intercept
slope, intercept = np.polyfit(log_eta, log_rho_eu_48_48, 1)
print(f"Slope: {slope}, Intercept: {intercept}")

# Create fit line
eta_fit = np.logspace(1, -12, 100)
rho_eu_48_48_fit = 10**(slope * np.log10(eta_fit) + intercept)

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(np.log10(eta_fit), np.log10(rho_eu_48_48_fit), label='Linear fit extrapolated', linewidth=3, zorder=1)
ax.plot(np.log10(eta), np.log10(rho_eu_48_48), label='Emu', marker='o', linewidth=3, markersize=12, zorder=2)
ax.scatter([np.log10(1e-5)], [np.log10(har_rho_eu_48_48)], label='EMU HAR', marker='x', s=200, c='black', zorder=3)
ax.scatter([np.log10(1e-5)], [np.log10(hsr_rho_eu_96_60)], label=r'EMU HSR + $\eta_\mathrm{matter}=1e-2$', marker='x', s=200, c='red', zorder=3)
ax.axhline(0, color='gray', linewidth=3, linestyle='--')
ax.axvline(0, color='gray', linewidth=3, linestyle='--')
ax.set_xlabel(r'$\log \eta$')
ax.set_ylabel(r'$\log \rho_{eu}$')
ax.set_title('Quantum coherence in the polar region at index z48')
leg = ax.legend(framealpha=0.0, ncol=1, fontsize=20)
pf.apply_custom_settings(ax, leg, log_scale_y=False)
plt.show()
plt.close(fig)

In [ ]:
eta = np.array([1e-5,1e-6,1e-7, 1e-8,1e-9,1e-10,1e-11,1e-12])
rho_eu_32_16 = np.array([2.0949725700233037e-09, 1.694172875250579e-09, 1.6314031395045477e-09, 5.590936886097058e-10, 6.189274147216006e-11, 6.195684651552251e-12, 6.195748792864933e-13, 6.195749434281654e-14])

log_eta = np.log10(eta)
log_rho_eu_32_16 = np.log10(rho_eu_32_16)

# Fit a straight line: log_rho_eu_32_16 = slope * log_eta + intercept
slope, intercept = np.polyfit(log_eta, log_rho_eu_32_16, 1)
# slope, intercept = np.polyfit(log_eta[0:4], log_rho_eu_32_16[0:4], 1)
print(f"Slope: {slope}, Intercept: {intercept}")

# Create fit line
eta_fit = np.logspace(1, -12, 100)
rho_eu_fit_32_16 = 10**(slope * np.log10(eta_fit) + intercept)

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(np.log10(eta_fit), np.log10(rho_eu_fit_32_16), label='Linear fit extrapolated', linewidth=3, zorder=1)
ax.plot(np.log10(eta), np.log10(rho_eu_32_16), label='Emu', marker='o', linewidth=3, markersize=12, zorder=2)
ax.scatter([np.log10(1e-5)], [np.log10(har_rho_eu_32_16)], label='EMU HAR', marker='x', s=200, c='black', zorder=3)
ax.scatter([np.log10(1e-5)], [np.log10(hsr_rho_eu_64_32)], label=r'EMU HSR + $\eta_\mathrm{matter}=1e-2$', marker='x', s=200, c='red', zorder=3)
ax.axhline(0, color='gray', linewidth=3, linestyle='--')
ax.axvline(0, color='gray', linewidth=3, linestyle='--')
ax.set_xlabel(r'$\log \eta$')
ax.set_ylabel(r'$\log \rho_{eu}$')
ax.set_title('Quantum coherence in the disk')
leg = ax.legend(framealpha=0.0, ncol=1, fontsize=20)
pf.apply_custom_settings(ax, leg, log_scale_y=False)
plt.show()
plt.close(fig)

In [ ]:
eta_48_60 = np.array([1e-12,1e-11,1e-10,1e-9,1e-8,1e-7,1e-6,1e-5])
rho_eu_48_60 = np.array([3.486066794093794e-13, 3.4860667707824836e-12, 3.486064439639409e-11, 3.485831200108597e-10, 3.461509739288298e-09, 2.9205195084108365e-08, 3.364449762494941e-07, 9.314665023912023e-06])

log_eta = np.log10(eta_48_60)
log_rho_eu_48_60 = np.log10(rho_eu_48_60)

# Fit a straight line: log_frequency = slope * log_eta + intercept
slope, intercept = np.polyfit(log_eta, log_rho_eu_48_60, 1)
print(f"Slope: {slope}, Intercept: {intercept}")

# Create fit line
eta_fit = np.logspace(1, -12, 100)
rho_eu_48_60_fit = 10**(slope * np.log10(eta_fit) + intercept)

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(np.log10(eta_fit), np.log10(rho_eu_48_60_fit), label='Linear fit extrapolated', linewidth=3, zorder=1)
ax.plot(np.log10(eta), np.log10(rho_eu_48_48), label='Emu', marker='o', linewidth=3, markersize=12, zorder=2)
ax.scatter([np.log10(1e-5)], [np.log10(har_rho_eu_48_48)], label='EMU HAR', marker='x', s=200, c='black', zorder=3)
ax.scatter([np.log10(1e-5)], [np.log10(hsr_rho_eu_96_60)], label=r'EMU HSR + $\eta_\mathrm{matter}=1e-2$', marker='x', s=200, c='red', zorder=3)
ax.axhline(0, color='gray', linewidth=3, linestyle='--')
ax.axvline(0, color='gray', linewidth=3, linestyle='--')
ax.set_xlabel(r'$\log \eta$')
ax.set_ylabel(r'$\log \rho_{eu}$')
ax.set_title('Quantum coherence in the polar region at index z60')
leg = ax.legend(framealpha=0.0, ncol=1, fontsize=20)
pf.apply_custom_settings(ax, leg, log_scale_y=False)
plt.show()
plt.close(fig)

### Adding effective mixing angles \sin 2 \theta_{eff} values and frequencies with homogeneous self interaction hamiltonian (just number densities: no flux term)

In [ ]:
eta_for_energy_plots = [1e-12,1e-11,1e-10,1e-9,1e-8,1e-7,1e-6,1e-5]
energybinsMeV_0_13 = [1.0, 3.0, 5.2382, 8.0097, 11.442, 15.691, 20.953, 27.468, 35.536, 45.525, 57.895, 73.212, 92.178]

In [ ]:
sin_2thetaeff_for_all_energies_48_48_att_total_1e_min_05_att_matter_1e_plus_00_att_nu_nu_1e_plus_00_att_vacuum_1e_plus_00 = [4.94917054593381e-06, 1.6497212761530736e-06, 9.448211969578011e-07, 6.178960171413183e-07, 4.3254337990272146e-07, 3.154140025943849e-07, 2.362029759676081e-07, 1.8017914816848613e-07, 1.3927174581408017e-07, 1.0871302981197559e-07, 8.548511333284635e-08, 6.76004018432889e-08, 5.369134260558445e-08]
sin_2thetaeff_for_all_energies_32_16_att_total_1e_min_05_att_matter_1e_plus_00_att_nu_nu_1e_plus_00_att_vacuum_1e_plus_00 = [2.544019762663205e-08, 8.480065661127664e-09, 4.856667635062959e-09, 3.1761736688577817e-09, 2.223404694031014e-09, 1.621324086438644e-09, 1.2141555653170958e-09, 9.261757031379962e-10, 7.158994622739915e-10, 5.588180033226668e-10, 4.3941973427316914e-10, 3.4748660649605843e-10, 2.7599002006703775e-10]

sin_2thetaeff_for_all_energies_48_48_att_total_1e_min_06_att_matter_1e_plus_00_att_nu_nu_1e_plus_00_att_vacuum_1e_plus_00 = [4.94917054593381e-06, 1.6497212761530736e-06, 9.448211969578011e-07, 6.178960171413183e-07, 4.3254337990272146e-07, 3.154140025943849e-07, 2.362029759676081e-07, 1.8017914816848613e-07, 1.3927174581408017e-07, 1.0871302981197559e-07, 8.548511333284635e-08, 6.76004018432889e-08, 5.369134260558445e-08]
sin_2thetaeff_for_all_energies_32_16_att_total_1e_min_06_att_matter_1e_plus_00_att_nu_nu_1e_plus_00_att_vacuum_1e_plus_00 = [2.544019762663205e-08, 8.480065661127664e-09, 4.856667635062959e-09, 3.1761736688577817e-09, 2.223404694031014e-09, 1.621324086438644e-09, 1.2141555653170958e-09, 9.261757031379962e-10, 7.158994622739915e-10, 5.588180033226668e-10, 4.3941973427316914e-10, 3.4748660649605843e-10, 2.7599002006703775e-10]

sin_2thetaeff_for_all_energies_48_48_att_total_1e_min_07_att_matter_1e_plus_00_att_nu_nu_1e_plus_00_att_vacuum_1e_plus_00 = [4.94917054593381e-06, 1.6497212761530736e-06, 9.448211969578011e-07, 6.178960171413183e-07, 4.3254337990272146e-07, 3.154140025943849e-07, 2.362029759676081e-07, 1.8017914816848613e-07, 1.3927174581408017e-07, 1.0871302981197559e-07, 8.548511333284635e-08, 6.76004018432889e-08, 5.369134260558445e-08]
sin_2thetaeff_for_all_energies_32_16_att_total_1e_min_07_att_matter_1e_plus_00_att_nu_nu_1e_plus_00_att_vacuum_1e_plus_00 = [2.544019762663205e-08, 8.480065661127664e-09, 4.856667635062959e-09, 3.1761736688577817e-09, 2.223404694031014e-09, 1.621324086438644e-09, 1.2141555653170958e-09, 9.261757031379962e-10, 7.158994622739915e-10, 5.588180033226668e-10, 4.3941973427316914e-10, 3.4748660649605843e-10, 2.7599002006703775e-10]

sin_2thetaeff_for_all_energies_48_48_att_total_1e_min_08_att_matter_1e_plus_00_att_nu_nu_1e_plus_00_att_vacuum_1e_plus_00 = [4.94917054593381e-06, 1.6497212761530736e-06, 9.448211969578011e-07, 6.178960171413183e-07, 4.3254337990272146e-07, 3.154140025943849e-07, 2.362029759676081e-07, 1.8017914816848613e-07, 1.3927174581408017e-07, 1.0871302981197559e-07, 8.548511333284635e-08, 6.76004018432889e-08, 5.369134260558445e-08]
sin_2thetaeff_for_all_energies_32_16_att_total_1e_min_08_att_matter_1e_plus_00_att_nu_nu_1e_plus_00_att_vacuum_1e_plus_00 = [2.544019762663205e-08, 8.480065661127664e-09, 4.856667635062959e-09, 3.1761736688577817e-09, 2.223404694031014e-09, 1.621324086438644e-09, 1.2141555653170958e-09, 9.261757031379962e-10, 7.158994622739915e-10, 5.588180033226668e-10, 4.3941973427316914e-10, 3.4748660649605843e-10, 2.7599002006703775e-10]

sin_2thetaeff_for_all_energies_48_48_att_total_1e_min_09_att_matter_1e_plus_00_att_nu_nu_1e_plus_00_att_vacuum_1e_plus_00 = [4.94917054593381e-06, 1.6497212761530736e-06, 9.448211969578011e-07, 6.178960171413183e-07, 4.3254337990272146e-07, 3.154140025943849e-07, 2.362029759676081e-07, 1.8017914816848613e-07, 1.3927174581408017e-07, 1.0871302981197559e-07, 8.548511333284635e-08, 6.76004018432889e-08, 5.369134260558445e-08]
sin_2thetaeff_for_all_energies_32_16_att_total_1e_min_09_att_matter_1e_plus_00_att_nu_nu_1e_plus_00_att_vacuum_1e_plus_00 = [2.544019762663205e-08, 8.480065661127664e-09, 4.856667635062959e-09, 3.1761736688577817e-09, 2.223404694031014e-09, 1.621324086438644e-09, 1.2141555653170958e-09, 9.261757031379962e-10, 7.158994622739915e-10, 5.588180033226668e-10, 4.3941973427316914e-10, 3.4748660649605843e-10, 2.7599002006703775e-10]

sin_2thetaeff_for_all_energies_48_48_att_total_1e_min_10_att_matter_1e_plus_00_att_nu_nu_1e_plus_00_att_vacuum_1e_plus_00 = [4.94917054593381e-06, 1.6497212761530736e-06, 9.448211969578011e-07, 6.178960171413183e-07, 4.3254337990272146e-07, 3.154140025943849e-07, 2.362029759676081e-07, 1.8017914816848613e-07, 1.3927174581408017e-07, 1.0871302981197559e-07, 8.548511333284635e-08, 6.76004018432889e-08, 5.369134260558445e-08]
sin_2thetaeff_for_all_energies_32_16_att_total_1e_min_10_att_matter_1e_plus_00_att_nu_nu_1e_plus_00_att_vacuum_1e_plus_00 = [2.544019762663205e-08, 8.480065661127664e-09, 4.856667635062959e-09, 3.1761736688577817e-09, 2.223404694031014e-09, 1.621324086438644e-09, 1.2141555653170958e-09, 9.261757031379962e-10, 7.158994622739915e-10, 5.588180033226668e-10, 4.3941973427316914e-10, 3.4748660649605843e-10, 2.7599002006703775e-10]

sin_2thetaeff_for_all_energies_48_48_att_total_1e_min_11_att_matter_1e_plus_00_att_nu_nu_1e_plus_00_att_vacuum_1e_plus_00 = [4.94917054593381e-06, 1.6497212761530736e-06, 9.448211969578011e-07, 6.178960171413183e-07, 4.3254337990272146e-07, 3.154140025943849e-07, 2.362029759676081e-07, 1.8017914816848613e-07, 1.3927174581408017e-07, 1.0871302981197559e-07, 8.548511333284635e-08, 6.76004018432889e-08, 5.369134260558445e-08]
sin_2thetaeff_for_all_energies_32_16_att_total_1e_min_11_att_matter_1e_plus_00_att_nu_nu_1e_plus_00_att_vacuum_1e_plus_00 = [2.544019762663205e-08, 8.480065661127664e-09, 4.856667635062959e-09, 3.1761736688577817e-09, 2.223404694031014e-09, 1.621324086438644e-09, 1.2141555653170958e-09, 9.261757031379962e-10, 7.158994622739915e-10, 5.588180033226668e-10, 4.3941973427316914e-10, 3.4748660649605843e-10, 2.7599002006703775e-10]

sin_2thetaeff_for_all_energies_48_48_att_total_1e_min_12_att_matter_1e_plus_00_att_nu_nu_1e_plus_00_att_vacuum_1e_plus_00 = [4.94917054593381e-06, 1.6497212761530736e-06, 9.448211969578011e-07, 6.178960171413183e-07, 4.3254337990272146e-07, 3.154140025943849e-07, 2.362029759676081e-07, 1.8017914816848613e-07, 1.3927174581408017e-07, 1.0871302981197559e-07, 8.548511333284635e-08, 6.76004018432889e-08, 5.369134260558445e-08]
sin_2thetaeff_for_all_energies_32_16_att_total_1e_min_12_att_matter_1e_plus_00_att_nu_nu_1e_plus_00_att_vacuum_1e_plus_00 = [2.544019762663205e-08, 8.480065661127664e-09, 4.856667635062959e-09, 3.1761736688577817e-09, 2.223404694031014e-09, 1.621324086438644e-09, 1.2141555653170958e-09, 9.261757031379962e-10, 7.158994622739915e-10, 5.588180033226668e-10, 4.3941973427316914e-10, 3.4748660649605843e-10, 2.7599002006703775e-10]

sin_2thetaeff_all_att_48_48 = np.array([
    sin_2thetaeff_for_all_energies_48_48_att_total_1e_min_12_att_matter_1e_plus_00_att_nu_nu_1e_plus_00_att_vacuum_1e_plus_00,
    sin_2thetaeff_for_all_energies_48_48_att_total_1e_min_11_att_matter_1e_plus_00_att_nu_nu_1e_plus_00_att_vacuum_1e_plus_00,
    sin_2thetaeff_for_all_energies_48_48_att_total_1e_min_10_att_matter_1e_plus_00_att_nu_nu_1e_plus_00_att_vacuum_1e_plus_00,
    sin_2thetaeff_for_all_energies_48_48_att_total_1e_min_09_att_matter_1e_plus_00_att_nu_nu_1e_plus_00_att_vacuum_1e_plus_00,
    sin_2thetaeff_for_all_energies_48_48_att_total_1e_min_08_att_matter_1e_plus_00_att_nu_nu_1e_plus_00_att_vacuum_1e_plus_00,
    sin_2thetaeff_for_all_energies_48_48_att_total_1e_min_07_att_matter_1e_plus_00_att_nu_nu_1e_plus_00_att_vacuum_1e_plus_00,
    sin_2thetaeff_for_all_energies_48_48_att_total_1e_min_06_att_matter_1e_plus_00_att_nu_nu_1e_plus_00_att_vacuum_1e_plus_00,
    sin_2thetaeff_for_all_energies_48_48_att_total_1e_min_05_att_matter_1e_plus_00_att_nu_nu_1e_plus_00_att_vacuum_1e_plus_00,
])

sin_2thetaeff_all_att_32_16 = np.array([
    sin_2thetaeff_for_all_energies_32_16_att_total_1e_min_12_att_matter_1e_plus_00_att_nu_nu_1e_plus_00_att_vacuum_1e_plus_00,
    sin_2thetaeff_for_all_energies_32_16_att_total_1e_min_11_att_matter_1e_plus_00_att_nu_nu_1e_plus_00_att_vacuum_1e_plus_00,
    sin_2thetaeff_for_all_energies_32_16_att_total_1e_min_10_att_matter_1e_plus_00_att_nu_nu_1e_plus_00_att_vacuum_1e_plus_00,
    sin_2thetaeff_for_all_energies_32_16_att_total_1e_min_09_att_matter_1e_plus_00_att_nu_nu_1e_plus_00_att_vacuum_1e_plus_00,
    sin_2thetaeff_for_all_energies_32_16_att_total_1e_min_08_att_matter_1e_plus_00_att_nu_nu_1e_plus_00_att_vacuum_1e_plus_00,
    sin_2thetaeff_for_all_energies_32_16_att_total_1e_min_07_att_matter_1e_plus_00_att_nu_nu_1e_plus_00_att_vacuum_1e_plus_00,
    sin_2thetaeff_for_all_energies_32_16_att_total_1e_min_06_att_matter_1e_plus_00_att_nu_nu_1e_plus_00_att_vacuum_1e_plus_00,
    sin_2thetaeff_for_all_energies_32_16_att_total_1e_min_05_att_matter_1e_plus_00_att_nu_nu_1e_plus_00_att_vacuum_1e_plus_00,
])

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
for i in range(len(energybinsMeV_0_13)):
    ax.plot(
        np.log10(eta_for_energy_plots),
        np.log10(sin_2thetaeff_all_att_48_48[:, i]),
        label=r'$\sin 2\theta_{\mathrm{eff}}$ for $E = %.3f$' % energybinsMeV_0_13[i],
        linewidth=3
    )
ax.plot(np.log10(eta), np.log10(rho_eu_48_48), label='Emu', marker='o', linewidth=3, markersize=12)
# ax.axhline(0, color='gray', linewidth=3, linestyle='--')
# ax.axvline(0, color='gray', linewidth=3, linestyle='--')
ax.set_xlabel(r'$\log \eta$')
ax.set_ylabel(r'$\log \rho_{eu}$')
ax.set_title('Quantum coherence in the poles')
leg = ax.legend(framealpha=0.0, ncol=2, fontsize=12)
pf.apply_custom_settings(ax, leg, log_scale_y=False)
plt.show()
plt.close(fig)

fig, ax = plt.subplots(figsize=(10, 6))
for i in range(len(energybinsMeV_0_13)):
    ax.plot(
        np.log10(eta_for_energy_plots),
        np.log10(sin_2thetaeff_all_att_32_16[:,i]),
        label=r'$\sin 2\theta_{\mathrm{eff}}$ for $E = %.3f$' % energybinsMeV_0_13[i],
        linewidth=3
    )
ax.plot(np.log10(eta), np.log10(rho_eu_32_16), label='Emu', marker='o', linewidth=3, markersize=12)
# ax.axhline(0, color='gray', linewidth=3, linestyle='--')
# ax.axvline(0, color='gray', linewidth=3, linestyle='--')
ax.set_xlabel(r'$\log \eta$')
ax.set_ylabel(r'$\log \rho_{eu}$')
ax.set_title('Quantum coherence in the disk')
leg = ax.legend(framealpha=0.0, ncol=2, fontsize=12)
pf.apply_custom_settings(ax, leg, log_scale_y=False)
plt.show()
plt.close(fig)

In [ ]:
frequency_invseconds_for_all_energies_48_48_att_total_1e_min_12_att_matter_1e_plus_00_att_nu_nu_1e_plus_00_att_vacuum_1e_plus_00 = [0.03295490665682332, 0.032954951384180003, 0.032954960939872505, 0.03295496537170877, 0.03295496788437307, 0.03295496947219481, 0.032954970545990586, 0.03295497130545762, 0.0329549718600043, 0.032954972274262744, 0.032954972589143816, 0.03295497283159163, 0.03295497302014492]
frequency_invseconds_for_all_energies_32_16_att_total_1e_min_12_att_matter_1e_plus_00_att_nu_nu_1e_plus_00_att_vacuum_1e_plus_00 = [6.41109225676465, 6.411092301492364, 6.411092311048087, 6.411092315479931, 6.4110923179925985, 6.411092319580422, 6.411092320654219, 6.411092321413685, 6.411092321968233, 6.4110923223824905, 6.411092322697373, 6.41109232293982, 6.411092323128373]

frequency_invseconds_for_all_energies_48_48_att_total_1e_min_11_att_matter_1e_plus_00_att_nu_nu_1e_plus_00_att_vacuum_1e_plus_00 = [0.32954906656823324, 0.3295495138418, 0.32954960939872496, 0.3295496537170877, 0.3295496788437306, 0.32954969472194806, 0.3295497054599059, 0.32954971305457625, 0.329549718600043, 0.32954972274262745, 0.32954972589143816, 0.32954972831591633, 0.3295497302014492]
frequency_invseconds_for_all_energies_32_16_att_total_1e_min_11_att_matter_1e_plus_00_att_nu_nu_1e_plus_00_att_vacuum_1e_plus_00 = [64.11092256764651, 64.11092301492366, 64.11092311048085, 64.11092315479931, 64.11092317992599, 64.11092319580422, 64.11092320654217, 64.11092321413685, 64.11092321968232, 64.11092322382491, 64.11092322697372, 64.1109232293982, 64.11092323128375]

frequency_invseconds_for_all_energies_48_48_att_total_1e_min_10_att_matter_1e_plus_00_att_nu_nu_1e_plus_00_att_vacuum_1e_plus_00 = [3.295490665682332, 3.295495138417999, 3.2954960939872504, 3.2954965371708775, 3.2954967884373065, 3.295496947219481, 3.2954970545990596, 3.2954971305457623, 3.2954971860004303, 3.2954972274262744, 3.295497258914381, 3.295497283159164, 3.295497302014492]
frequency_invseconds_for_all_energies_32_16_att_total_1e_min_10_att_matter_1e_plus_00_att_nu_nu_1e_plus_00_att_vacuum_1e_plus_00 = [641.1092256764654, 641.1092301492365, 641.1092311048087, 641.1092315479931, 641.1092317992599, 641.1092319580423, 641.1092320654219, 641.1092321413686, 641.1092321968232, 641.1092322382491, 641.1092322697374, 641.109232293982, 641.1092323128373]

frequency_invseconds_for_all_energies_48_48_att_total_1e_min_09_att_matter_1e_plus_00_att_nu_nu_1e_plus_00_att_vacuum_1e_plus_00 = [32.95490665682332, 32.95495138418, 32.95496093987251, 32.95496537170877, 32.95496788437307, 32.95496947219481, 32.95497054599059, 32.95497130545762, 32.9549718600043, 32.954972274262744, 32.95497258914382, 32.95497283159164, 32.95497302014492]
frequency_invseconds_for_all_energies_32_16_att_total_1e_min_09_att_matter_1e_plus_00_att_nu_nu_1e_plus_00_att_vacuum_1e_plus_00 = [6411.092256764651, 6411.092301492365, 6411.092311048087, 6411.092315479931, 6411.092317992599, 6411.092319580423, 6411.092320654219, 6411.092321413686, 6411.092321968233, 6411.092322382492, 6411.092322697373, 6411.09232293982, 6411.092323128373]

frequency_invseconds_for_all_energies_48_48_att_total_1e_min_08_att_matter_1e_plus_00_att_nu_nu_1e_plus_00_att_vacuum_1e_plus_00 = [329.5490665682331, 329.54951384180004, 329.54960939872507, 329.5496537170877, 329.54967884373065, 329.5496947219481, 329.5497054599059, 329.54971305457616, 329.54971860004304, 329.5497227426275, 329.5497258914382, 329.5497283159163, 329.54973020144917]
frequency_invseconds_for_all_energies_32_16_att_total_1e_min_08_att_matter_1e_plus_00_att_nu_nu_1e_plus_00_att_vacuum_1e_plus_00 = [64110.9225676465, 64110.923014923654, 64110.92311048087, 64110.92315479931, 64110.92317992598, 64110.92319580422, 64110.92320654219, 64110.92321413686, 64110.92321968233, 64110.92322382491, 64110.92322697374, 64110.9232293982, 64110.923231283734]

frequency_invseconds_for_all_energies_48_48_att_total_1e_min_07_att_matter_1e_plus_00_att_nu_nu_1e_plus_00_att_vacuum_1e_plus_00 = [3295.4906656823314, 3295.495138418, 3295.4960939872503, 3295.496537170877, 3295.496788437306, 3295.49694721948, 3295.4970545990586, 3295.497130545762, 3295.4971860004307, 3295.4972274262736, 3295.4972589143813, 3295.4972831591635, 3295.4973020144917]
frequency_invseconds_for_all_energies_32_16_att_total_1e_min_07_att_matter_1e_plus_00_att_nu_nu_1e_plus_00_att_vacuum_1e_plus_00 = [641109.225676465, 641109.2301492364, 641109.2311048087, 641109.2315479931, 641109.2317992598, 641109.2319580422, 641109.232065422, 641109.2321413686, 641109.2321968232, 641109.2322382492, 641109.2322697373, 641109.232293982, 641109.2323128373]

frequency_invseconds_for_all_energies_48_48_att_total_1e_min_06_att_matter_1e_plus_00_att_nu_nu_1e_plus_00_att_vacuum_1e_plus_00 = [32954.90665682332, 32954.95138417999, 32954.9609398725, 32954.96537170877, 32954.967884373065, 32954.96947219481, 32954.970545990596, 32954.97130545761, 32954.9718600043, 32954.972274262735, 32954.972589143814, 32954.972831591636, 32954.97302014492]
frequency_invseconds_for_all_energies_32_16_att_total_1e_min_06_att_matter_1e_plus_00_att_nu_nu_1e_plus_00_att_vacuum_1e_plus_00 = [6411092.25676465, 6411092.301492364, 6411092.311048087, 6411092.315479931, 6411092.317992599, 6411092.319580421, 6411092.320654219, 6411092.321413686, 6411092.321968232, 6411092.322382492, 6411092.322697373, 6411092.322939821, 6411092.323128372]

frequency_invseconds_for_all_energies_48_48_att_total_1e_min_05_att_matter_1e_plus_00_att_nu_nu_1e_plus_00_att_vacuum_1e_plus_00 = [329549.0665682332, 329549.51384180004, 329549.6093987251, 329549.65371708776, 329549.67884373065, 329549.69472194806, 329549.7054599059, 329549.71305457625, 329549.7186000431, 329549.7227426275, 329549.7258914382, 329549.7283159163, 329549.7302014491]
frequency_invseconds_for_all_energies_32_16_att_total_1e_min_05_att_matter_1e_plus_00_att_nu_nu_1e_plus_00_att_vacuum_1e_plus_00 = [64110922.56764651, 64110923.014923654, 64110923.11048088, 64110923.15479932, 64110923.17992599, 64110923.19580423, 64110923.206542194, 64110923.214136854, 64110923.219682336, 64110923.22382493, 64110923.22697374, 64110923.22939821, 64110923.23128374]

frequency_invseconds_all_att_48_48 = np.array([
    frequency_invseconds_for_all_energies_48_48_att_total_1e_min_12_att_matter_1e_plus_00_att_nu_nu_1e_plus_00_att_vacuum_1e_plus_00,
    frequency_invseconds_for_all_energies_48_48_att_total_1e_min_11_att_matter_1e_plus_00_att_nu_nu_1e_plus_00_att_vacuum_1e_plus_00,
    frequency_invseconds_for_all_energies_48_48_att_total_1e_min_10_att_matter_1e_plus_00_att_nu_nu_1e_plus_00_att_vacuum_1e_plus_00,
    frequency_invseconds_for_all_energies_48_48_att_total_1e_min_09_att_matter_1e_plus_00_att_nu_nu_1e_plus_00_att_vacuum_1e_plus_00,
    frequency_invseconds_for_all_energies_48_48_att_total_1e_min_08_att_matter_1e_plus_00_att_nu_nu_1e_plus_00_att_vacuum_1e_plus_00,
    frequency_invseconds_for_all_energies_48_48_att_total_1e_min_07_att_matter_1e_plus_00_att_nu_nu_1e_plus_00_att_vacuum_1e_plus_00,
    frequency_invseconds_for_all_energies_48_48_att_total_1e_min_06_att_matter_1e_plus_00_att_nu_nu_1e_plus_00_att_vacuum_1e_plus_00,
    frequency_invseconds_for_all_energies_48_48_att_total_1e_min_05_att_matter_1e_plus_00_att_nu_nu_1e_plus_00_att_vacuum_1e_plus_00,
])

frequency_invseconds_all_att_32_16 = np.array([
    frequency_invseconds_for_all_energies_32_16_att_total_1e_min_12_att_matter_1e_plus_00_att_nu_nu_1e_plus_00_att_vacuum_1e_plus_00,
    frequency_invseconds_for_all_energies_32_16_att_total_1e_min_11_att_matter_1e_plus_00_att_nu_nu_1e_plus_00_att_vacuum_1e_plus_00,
    frequency_invseconds_for_all_energies_32_16_att_total_1e_min_10_att_matter_1e_plus_00_att_nu_nu_1e_plus_00_att_vacuum_1e_plus_00,
    frequency_invseconds_for_all_energies_32_16_att_total_1e_min_09_att_matter_1e_plus_00_att_nu_nu_1e_plus_00_att_vacuum_1e_plus_00,
    frequency_invseconds_for_all_energies_32_16_att_total_1e_min_08_att_matter_1e_plus_00_att_nu_nu_1e_plus_00_att_vacuum_1e_plus_00,
    frequency_invseconds_for_all_energies_32_16_att_total_1e_min_07_att_matter_1e_plus_00_att_nu_nu_1e_plus_00_att_vacuum_1e_plus_00,
    frequency_invseconds_for_all_energies_32_16_att_total_1e_min_06_att_matter_1e_plus_00_att_nu_nu_1e_plus_00_att_vacuum_1e_plus_00,
    frequency_invseconds_for_all_energies_32_16_att_total_1e_min_05_att_matter_1e_plus_00_att_nu_nu_1e_plus_00_att_vacuum_1e_plus_00,
])

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
for i in range(len(energybinsMeV_0_13)):
    ax.plot(np.log10(eta_for_energy_plots), np.log10(frequency_invseconds_all_att_48_48[:,i]/pc.PhysConst.c), label=f'$E = {energybinsMeV_0_13[i]}$', linewidth=3)
# ax.axhline(0, color='gray', linewidth=3, linestyle='--')
# ax.axvline(0, color='gray', linewidth=3, linestyle='--')
ax.set_xlabel(r'$\log \eta$')
ax.set_ylabel(r'$\log\,\omega\,[\,\mathrm{cm}^{-1}\,]$')
ax.set_title('Effective frequency in the poles')
leg = ax.legend(framealpha=0.0, ncol=1, fontsize=12)
pf.apply_custom_settings(ax, leg, log_scale_y=False)
plt.show()
plt.close(fig)

fig, ax = plt.subplots(figsize=(10, 6))
for i in range(len(energybinsMeV_0_13)):
    ax.plot(np.log10(eta_for_energy_plots), np.log10(frequency_invseconds_all_att_32_16[:,i]/pc.PhysConst.c), label=f'$E = {energybinsMeV_0_13[i]}$', linewidth=3)
# ax.axhline(0, color='gray', linewidth=3, linestyle='--')
# ax.axvline(0, color='gray', linewidth=3, linestyle='--')
ax.set_xlabel(r'$\log \eta$')
ax.set_ylabel(r'$\log\,\omega\,[\,\mathrm{cm}^{-1}\,]$')
ax.set_title('Effective frequency in the disk')
leg = ax.legend(framealpha=0.0, ncol=1, fontsize=12)
pf.apply_custom_settings(ax, leg, log_scale_y=False)
plt.show()
plt.close(fig)

# Adding effective mixing angles \sin 2 \theta_{eff} values and frequencies with self interaction hamiltonian with flux correction in the pole: with phat = zhat so only z component of flux is included

In [ ]:
sin_2thetaeff_fluxz_for_all_energies_48_48_att_total_1e_min_05_att_matter_1e_plus_00_att_nu_nu_1e_plus_00_att_vacuum_1e_plus_00 = [1.5901607112297144e-05, 5.30051259027603e-06, 3.0356845780522634e-06, 1.9852823559642148e-06, 1.3897493610469695e-06, 1.0134159534992972e-06, 7.589131897470133e-07, 5.789102549672323e-07, 4.474759688458595e-07, 3.4929171611774997e-07, 2.746611086460496e-07, 2.1719806188070216e-07, 1.725086705820562e-07]
sin_2thetaeff_fluxz_for_all_energies_48_48_att_total_1e_min_06_att_matter_1e_plus_00_att_nu_nu_1e_plus_00_att_vacuum_1e_plus_00 = [1.5901607112297144e-05, 5.30051259027603e-06, 3.0356845780522634e-06, 1.9852823559642148e-06, 1.3897493610469695e-06, 1.0134159534992972e-06, 7.589131897470133e-07, 5.789102549672323e-07, 4.474759688458595e-07, 3.4929171611774997e-07, 2.746611086460496e-07, 2.1719806188070216e-07, 1.725086705820562e-07]
sin_2thetaeff_fluxz_for_all_energies_48_48_att_total_1e_min_07_att_matter_1e_plus_00_att_nu_nu_1e_plus_00_att_vacuum_1e_plus_00 = [1.5901607112297144e-05, 5.30051259027603e-06, 3.0356845780522634e-06, 1.9852823559642148e-06, 1.3897493610469695e-06, 1.0134159534992972e-06, 7.589131897470133e-07, 5.789102549672323e-07, 4.474759688458595e-07, 3.4929171611774997e-07, 2.746611086460496e-07, 2.1719806188070216e-07, 1.725086705820562e-07]
sin_2thetaeff_fluxz_for_all_energies_48_48_att_total_1e_min_08_att_matter_1e_plus_00_att_nu_nu_1e_plus_00_att_vacuum_1e_plus_00 = [1.5901607112297144e-05, 5.30051259027603e-06, 3.0356845780522634e-06, 1.9852823559642148e-06, 1.3897493610469695e-06, 1.0134159534992972e-06, 7.589131897470133e-07, 5.789102549672323e-07, 4.474759688458595e-07, 3.4929171611774997e-07, 2.746611086460496e-07, 2.1719806188070216e-07, 1.725086705820562e-07]
sin_2thetaeff_fluxz_for_all_energies_48_48_att_total_1e_min_09_att_matter_1e_plus_00_att_nu_nu_1e_plus_00_att_vacuum_1e_plus_00 = [1.5901607112297144e-05, 5.30051259027603e-06, 3.0356845780522634e-06, 1.9852823559642148e-06, 1.3897493610469695e-06, 1.0134159534992972e-06, 7.589131897470133e-07, 5.789102549672323e-07, 4.474759688458595e-07, 3.4929171611774997e-07, 2.746611086460496e-07, 2.1719806188070216e-07, 1.725086705820562e-07]
sin_2thetaeff_fluxz_for_all_energies_48_48_att_total_1e_min_10_att_matter_1e_plus_00_att_nu_nu_1e_plus_00_att_vacuum_1e_plus_00 = [1.5901607112297144e-05, 5.30051259027603e-06, 3.0356845780522634e-06, 1.9852823559642148e-06, 1.3897493610469695e-06, 1.0134159534992972e-06, 7.589131897470133e-07, 5.789102549672323e-07, 4.474759688458595e-07, 3.4929171611774997e-07, 2.746611086460496e-07, 2.1719806188070216e-07, 1.725086705820562e-07]
sin_2thetaeff_fluxz_for_all_energies_48_48_att_total_1e_min_11_att_matter_1e_plus_00_att_nu_nu_1e_plus_00_att_vacuum_1e_plus_00 = [1.5901607112297144e-05, 5.30051259027603e-06, 3.0356845780522634e-06, 1.9852823559642148e-06, 1.3897493610469695e-06, 1.0134159534992972e-06, 7.589131897470133e-07, 5.789102549672323e-07, 4.474759688458595e-07, 3.4929171611774997e-07, 2.746611086460496e-07, 2.1719806188070216e-07, 1.725086705820562e-07]
sin_2thetaeff_fluxz_for_all_energies_48_48_att_total_1e_min_12_att_matter_1e_plus_00_att_nu_nu_1e_plus_00_att_vacuum_1e_plus_00 = [1.5901607112297144e-05, 5.30051259027603e-06, 3.0356845780522634e-06, 1.9852823559642148e-06, 1.3897493610469695e-06, 1.0134159534992972e-06, 7.589131897470133e-07, 5.789102549672323e-07, 4.474759688458595e-07, 3.4929171611774997e-07, 2.746611086460496e-07, 2.1719806188070216e-07, 1.725086705820562e-07]

sin_2thetaeff_fluxz_all_att_48_48 = np.array([
    sin_2thetaeff_fluxz_for_all_energies_48_48_att_total_1e_min_12_att_matter_1e_plus_00_att_nu_nu_1e_plus_00_att_vacuum_1e_plus_00,
    sin_2thetaeff_fluxz_for_all_energies_48_48_att_total_1e_min_11_att_matter_1e_plus_00_att_nu_nu_1e_plus_00_att_vacuum_1e_plus_00,
    sin_2thetaeff_fluxz_for_all_energies_48_48_att_total_1e_min_10_att_matter_1e_plus_00_att_nu_nu_1e_plus_00_att_vacuum_1e_plus_00,
    sin_2thetaeff_fluxz_for_all_energies_48_48_att_total_1e_min_09_att_matter_1e_plus_00_att_nu_nu_1e_plus_00_att_vacuum_1e_plus_00,
    sin_2thetaeff_fluxz_for_all_energies_48_48_att_total_1e_min_08_att_matter_1e_plus_00_att_nu_nu_1e_plus_00_att_vacuum_1e_plus_00,
    sin_2thetaeff_fluxz_for_all_energies_48_48_att_total_1e_min_07_att_matter_1e_plus_00_att_nu_nu_1e_plus_00_att_vacuum_1e_plus_00,
    sin_2thetaeff_fluxz_for_all_energies_48_48_att_total_1e_min_06_att_matter_1e_plus_00_att_nu_nu_1e_plus_00_att_vacuum_1e_plus_00,
    sin_2thetaeff_fluxz_for_all_energies_48_48_att_total_1e_min_05_att_matter_1e_plus_00_att_nu_nu_1e_plus_00_att_vacuum_1e_plus_00,
])

frequency_invseconds_fluxz_for_all_energies_48_48_att_total_1e_min_05_att_matter_1e_plus_00_att_nu_nu_1e_plus_00_att_vacuum_1e_plus_00 = [102567.90538222229, 102568.35264784981, 102568.44820410802, 102568.49252228436, 102568.51764885629, 102568.5335270418, 102568.54426498365, 102568.55185964543, 102568.55740510755, 102568.56154768915, 102568.56469649827, 102568.56712097545, 102568.56900650766]
frequency_invseconds_fluxz_for_all_energies_48_48_att_total_1e_min_06_att_matter_1e_plus_00_att_nu_nu_1e_plus_00_att_vacuum_1e_plus_00 = [10256.790538222225, 10256.83526478498, 10256.8448204108, 10256.849252228436, 10256.851764885627, 10256.853352704176, 10256.854426498365, 10256.855185964543, 10256.855740510757, 10256.856154768915, 10256.856469649829, 10256.856712097544, 10256.856900650766]
frequency_invseconds_fluxz_for_all_energies_48_48_att_total_1e_min_07_att_matter_1e_plus_00_att_nu_nu_1e_plus_00_att_vacuum_1e_plus_00 = [1025.6790538222226, 1025.683526478498, 1025.68448204108, 1025.6849252228435, 1025.6851764885628, 1025.6853352704177, 1025.6854426498367, 1025.6855185964541, 1025.6855740510753, 1025.6856154768916, 1025.6856469649827, 1025.6856712097544, 1025.6856900650766]
frequency_invseconds_fluxz_for_all_energies_48_48_att_total_1e_min_08_att_matter_1e_plus_00_att_nu_nu_1e_plus_00_att_vacuum_1e_plus_00 = [102.56790538222226, 102.5683526478498, 102.568448204108, 102.56849252228437, 102.56851764885629, 102.5685335270418, 102.56854426498366, 102.56855185964542, 102.56855740510755, 102.56856154768916, 102.56856469649826, 102.56856712097544, 102.56856900650766]
frequency_invseconds_fluxz_for_all_energies_48_48_att_total_1e_min_09_att_matter_1e_plus_00_att_nu_nu_1e_plus_00_att_vacuum_1e_plus_00 = [10.256790538222226, 10.25683526478498, 10.2568448204108, 10.256849252228438, 10.256851764885628, 10.256853352704178, 10.256854426498366, 10.256855185964543, 10.256855740510755, 10.256856154768917, 10.25685646964983, 10.256856712097544, 10.256856900650769]
frequency_invseconds_fluxz_for_all_energies_48_48_att_total_1e_min_10_att_matter_1e_plus_00_att_nu_nu_1e_plus_00_att_vacuum_1e_plus_00 = [1.0256790538222227, 1.025683526478498, 1.02568448204108, 1.0256849252228437, 1.0256851764885628, 1.025685335270418, 1.0256854426498367, 1.0256855185964542, 1.0256855740510755, 1.0256856154768916, 1.0256856469649829, 1.0256856712097544, 1.0256856900650768]
frequency_invseconds_fluxz_for_all_energies_48_48_att_total_1e_min_11_att_matter_1e_plus_00_att_nu_nu_1e_plus_00_att_vacuum_1e_plus_00 = [0.10256790538222227, 0.10256835264784982, 0.102568448204108, 0.10256849252228437, 0.10256851764885626, 0.10256853352704177, 0.10256854426498364, 0.10256855185964542, 0.10256855740510754, 0.10256856154768915, 0.10256856469649826, 0.10256856712097542, 0.10256856900650767]
frequency_invseconds_fluxz_for_all_energies_48_48_att_total_1e_min_12_att_matter_1e_plus_00_att_nu_nu_1e_plus_00_att_vacuum_1e_plus_00 = [0.010256790538222225, 0.010256835264784981, 0.010256844820410801, 0.010256849252228438, 0.010256851764885628, 0.01025685335270418, 0.010256854426498365, 0.010256855185964543, 0.010256855740510754, 0.010256856154768916, 0.010256856469649824, 0.010256856712097542, 0.010256856900650767]

frequency_invseconds_fluxz_all_att_48_48 = np.array([
    frequency_invseconds_fluxz_for_all_energies_48_48_att_total_1e_min_12_att_matter_1e_plus_00_att_nu_nu_1e_plus_00_att_vacuum_1e_plus_00,
    frequency_invseconds_fluxz_for_all_energies_48_48_att_total_1e_min_11_att_matter_1e_plus_00_att_nu_nu_1e_plus_00_att_vacuum_1e_plus_00,
    frequency_invseconds_fluxz_for_all_energies_48_48_att_total_1e_min_10_att_matter_1e_plus_00_att_nu_nu_1e_plus_00_att_vacuum_1e_plus_00,
    frequency_invseconds_fluxz_for_all_energies_48_48_att_total_1e_min_09_att_matter_1e_plus_00_att_nu_nu_1e_plus_00_att_vacuum_1e_plus_00,
    frequency_invseconds_fluxz_for_all_energies_48_48_att_total_1e_min_08_att_matter_1e_plus_00_att_nu_nu_1e_plus_00_att_vacuum_1e_plus_00,
    frequency_invseconds_fluxz_for_all_energies_48_48_att_total_1e_min_07_att_matter_1e_plus_00_att_nu_nu_1e_plus_00_att_vacuum_1e_plus_00,
    frequency_invseconds_fluxz_for_all_energies_48_48_att_total_1e_min_06_att_matter_1e_plus_00_att_nu_nu_1e_plus_00_att_vacuum_1e_plus_00,
    frequency_invseconds_fluxz_for_all_energies_48_48_att_total_1e_min_05_att_matter_1e_plus_00_att_nu_nu_1e_plus_00_att_vacuum_1e_plus_00,
])

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
for i in range(len(energybinsMeV_0_13)):
    ax.plot(
        np.log10(eta_for_energy_plots),
        np.log10(sin_2thetaeff_fluxz_all_att_48_48[:, i]),
        label=r'$\sin 2\theta_{\mathrm{eff}}$ for $E = %.3f$' % energybinsMeV_0_13[i],
        linewidth=3
    )
ax.plot(np.log10(eta), np.log10(rho_eu_48_48), label='Emu', marker='o', linewidth=3, markersize=12)
# ax.axhline(0, color='gray', linewidth=3, linestyle='--')
# ax.axvline(0, color='gray', linewidth=3, linestyle='--')
ax.set_xlabel(r'$\log \eta$')
ax.set_ylabel(r'$\log \rho_{eu}$')
ax.set_title('Quantum coherence in the poles \nwith self interaction hamiltonian along $\\hat{p}=\\hat{z}$')
leg = ax.legend(framealpha=0.0, ncol=2, fontsize=12)
pf.apply_custom_settings(ax, leg, log_scale_y=False)
plt.show()
plt.close(fig)

fig, ax = plt.subplots(figsize=(10, 6))
for i in range(len(energybinsMeV_0_13)):
    ax.plot(np.log10(eta_for_energy_plots), np.log10(frequency_invseconds_fluxz_all_att_48_48[:,i]/pc.PhysConst.c), label=f'$E = {energybinsMeV_0_13[i]}$', linewidth=3)
# ax.axhline(0, color='gray', linewidth=3, linestyle='--')
# ax.axvline(0, color='gray', linewidth=3, linestyle='--')
ax.set_xlabel(r'$\log \eta$')
ax.set_ylabel(r'$\log\,\omega\,[\,\mathrm{cm}^{-1}\,]$')
ax.set_title('Effective frequency in the poles \nwith self interaction hamiltonian along $\\hat{p}=\\hat{z}$')
leg = ax.legend(framealpha=0.0, ncol=1, fontsize=12)
pf.apply_custom_settings(ax, leg, log_scale_y=False)
plt.show()
plt.close(fig)

### Adding effective mixing angles \sin 2 \theta_{eff} values and frequencies with self interaction hamiltonian with flux correction in the disk: with phat = xhat so only x component of flux is included

In [ ]:
sin_2thetaeff_fluxx_for_all_energies_32_16_att_total_1e_min_05_att_matter_1e_plus_00_att_nu_nu_1e_plus_00_att_vacuum_1e_plus_00 = [2.5517409641191842e-08, 8.505803295373735e-09, 4.871407844116302e-09, 3.185813513335997e-09, 2.2301530736638956e-09, 1.6262450389729925e-09, 1.2178401734912218e-09, 9.289867878363471e-10, 7.18071946688578e-10, 5.60514424104294e-10, 4.407533341703489e-10, 3.4854131836945233e-10, 2.7682757231681497e-10]
sin_2thetaeff_fluxx_for_all_energies_32_16_att_total_1e_min_06_att_matter_1e_plus_00_att_nu_nu_1e_plus_00_att_vacuum_1e_plus_00 = [2.5517409641191842e-08, 8.505803295373735e-09, 4.871407844116302e-09, 3.185813513335997e-09, 2.2301530736638956e-09, 1.6262450389729925e-09, 1.2178401734912218e-09, 9.289867878363471e-10, 7.18071946688578e-10, 5.60514424104294e-10, 4.407533341703489e-10, 3.4854131836945233e-10, 2.7682757231681497e-10]
sin_2thetaeff_fluxx_for_all_energies_32_16_att_total_1e_min_07_att_matter_1e_plus_00_att_nu_nu_1e_plus_00_att_vacuum_1e_plus_00 = [2.5517409641191842e-08, 8.505803295373735e-09, 4.871407844116302e-09, 3.185813513335997e-09, 2.2301530736638956e-09, 1.6262450389729925e-09, 1.2178401734912218e-09, 9.289867878363471e-10, 7.18071946688578e-10, 5.60514424104294e-10, 4.407533341703489e-10, 3.4854131836945233e-10, 2.7682757231681497e-10]
sin_2thetaeff_fluxx_for_all_energies_32_16_att_total_1e_min_08_att_matter_1e_plus_00_att_nu_nu_1e_plus_00_att_vacuum_1e_plus_00 = [2.5517409641191842e-08, 8.505803295373735e-09, 4.871407844116302e-09, 3.185813513335997e-09, 2.2301530736638956e-09, 1.6262450389729925e-09, 1.2178401734912218e-09, 9.289867878363471e-10, 7.18071946688578e-10, 5.60514424104294e-10, 4.407533341703489e-10, 3.4854131836945233e-10, 2.7682757231681497e-10]
sin_2thetaeff_fluxx_for_all_energies_32_16_att_total_1e_min_09_att_matter_1e_plus_00_att_nu_nu_1e_plus_00_att_vacuum_1e_plus_00 = [2.5517409641191842e-08, 8.505803295373735e-09, 4.871407844116302e-09, 3.185813513335997e-09, 2.2301530736638956e-09, 1.6262450389729925e-09, 1.2178401734912218e-09, 9.289867878363471e-10, 7.18071946688578e-10, 5.60514424104294e-10, 4.407533341703489e-10, 3.4854131836945233e-10, 2.7682757231681497e-10]
sin_2thetaeff_fluxx_for_all_energies_32_16_att_total_1e_min_10_att_matter_1e_plus_00_att_nu_nu_1e_plus_00_att_vacuum_1e_plus_00 = [2.5517409641191842e-08, 8.505803295373735e-09, 4.871407844116302e-09, 3.185813513335997e-09, 2.2301530736638956e-09, 1.6262450389729925e-09, 1.2178401734912218e-09, 9.289867878363471e-10, 7.18071946688578e-10, 5.60514424104294e-10, 4.407533341703489e-10, 3.4854131836945233e-10, 2.7682757231681497e-10]
sin_2thetaeff_fluxx_for_all_energies_32_16_att_total_1e_min_11_att_matter_1e_plus_00_att_nu_nu_1e_plus_00_att_vacuum_1e_plus_00 = [2.5517409641191842e-08, 8.505803295373735e-09, 4.871407844116302e-09, 3.185813513335997e-09, 2.2301530736638956e-09, 1.6262450389729925e-09, 1.2178401734912218e-09, 9.289867878363471e-10, 7.18071946688578e-10, 5.60514424104294e-10, 4.407533341703489e-10, 3.4854131836945233e-10, 2.7682757231681497e-10]
sin_2thetaeff_fluxx_for_all_energies_32_16_att_total_1e_min_12_att_matter_1e_plus_00_att_nu_nu_1e_plus_00_att_vacuum_1e_plus_00 = [2.5517409641191842e-08, 8.505803295373735e-09, 4.871407844116302e-09, 3.185813513335997e-09, 2.2301530736638956e-09, 1.6262450389729925e-09, 1.2178401734912218e-09, 9.289867878363471e-10, 7.18071946688578e-10, 5.60514424104294e-10, 4.407533341703489e-10, 3.4854131836945233e-10, 2.7682757231681497e-10]


sin_2thetaeff_fluxx_all_att_32_16 = np.array([
    sin_2thetaeff_fluxx_for_all_energies_32_16_att_total_1e_min_12_att_matter_1e_plus_00_att_nu_nu_1e_plus_00_att_vacuum_1e_plus_00,
    sin_2thetaeff_fluxx_for_all_energies_32_16_att_total_1e_min_11_att_matter_1e_plus_00_att_nu_nu_1e_plus_00_att_vacuum_1e_plus_00,
    sin_2thetaeff_fluxx_for_all_energies_32_16_att_total_1e_min_10_att_matter_1e_plus_00_att_nu_nu_1e_plus_00_att_vacuum_1e_plus_00,
    sin_2thetaeff_fluxx_for_all_energies_32_16_att_total_1e_min_09_att_matter_1e_plus_00_att_nu_nu_1e_plus_00_att_vacuum_1e_plus_00,
    sin_2thetaeff_fluxx_for_all_energies_32_16_att_total_1e_min_08_att_matter_1e_plus_00_att_nu_nu_1e_plus_00_att_vacuum_1e_plus_00,
    sin_2thetaeff_fluxx_for_all_energies_32_16_att_total_1e_min_07_att_matter_1e_plus_00_att_nu_nu_1e_plus_00_att_vacuum_1e_plus_00,
    sin_2thetaeff_fluxx_for_all_energies_32_16_att_total_1e_min_06_att_matter_1e_plus_00_att_nu_nu_1e_plus_00_att_vacuum_1e_plus_00,
    sin_2thetaeff_fluxx_for_all_energies_32_16_att_total_1e_min_05_att_matter_1e_plus_00_att_nu_nu_1e_plus_00_att_vacuum_1e_plus_00,
])

frequency_invseconds_fluxx_for_all_energies_32_16_att_total_1e_min_05_att_matter_1e_plus_00_att_nu_nu_1e_plus_00_att_vacuum_1e_plus_00 = [63916932.33306099, 63916932.78033813, 63916932.87589535, 63916932.92021379, 63916932.94534048, 63916932.96121871, 63916932.97195667, 63916932.979551345, 63916932.98509681, 63916932.98923941, 63916932.99238822, 63916932.99481268, 63916932.996698216]
frequency_invseconds_fluxx_for_all_energies_32_16_att_total_1e_min_06_att_matter_1e_plus_00_att_nu_nu_1e_plus_00_att_vacuum_1e_plus_00 = [6391693.233306097, 6391693.278033812, 6391693.287589534, 6391693.292021378, 6391693.294534045, 6391693.296121869, 6391693.2971956665, 6391693.297955134, 6391693.29850968, 6391693.298923939, 6391693.29923882, 6391693.299481267, 6391693.299669821]
frequency_invseconds_fluxx_for_all_energies_32_16_att_total_1e_min_07_att_matter_1e_plus_00_att_nu_nu_1e_plus_00_att_vacuum_1e_plus_00 = [639169.3233306098, 639169.3278033811, 639169.3287589535, 639169.3292021378, 639169.3294534044, 639169.329612187, 639169.3297195666, 639169.3297955132, 639169.329850968, 639169.3298923939, 639169.3299238819, 639169.3299481267, 639169.329966982]
frequency_invseconds_fluxx_for_all_energies_32_16_att_total_1e_min_08_att_matter_1e_plus_00_att_nu_nu_1e_plus_00_att_vacuum_1e_plus_00 = [63916.93233306098, 63916.93278033811, 63916.93287589535, 63916.93292021378, 63916.93294534045, 63916.93296121869, 63916.93297195665, 63916.93297955133, 63916.9329850968, 63916.932989239394, 63916.93299238821, 63916.932994812676, 63916.93299669821]
frequency_invseconds_fluxx_for_all_energies_32_16_att_total_1e_min_09_att_matter_1e_plus_00_att_nu_nu_1e_plus_00_att_vacuum_1e_plus_00 = [6391.693233306099, 6391.693278033814, 6391.6932875895345, 6391.6932920213785, 6391.693294534047, 6391.69329612187, 6391.693297195667, 6391.693297955133, 6391.69329850968, 6391.693298923939, 6391.69329923882, 6391.693299481268, 6391.6932996698215]
frequency_invseconds_fluxx_for_all_energies_32_16_att_total_1e_min_10_att_matter_1e_plus_00_att_nu_nu_1e_plus_00_att_vacuum_1e_plus_00 = [639.1693233306098, 639.1693278033812, 639.1693287589534, 639.1693292021379, 639.1693294534047, 639.169329612187, 639.1693297195667, 639.1693297955134, 639.169329850968, 639.1693298923939, 639.1693299238821, 639.1693299481267, 639.1693299669821]
frequency_invseconds_fluxx_for_all_energies_32_16_att_total_1e_min_11_att_matter_1e_plus_00_att_nu_nu_1e_plus_00_att_vacuum_1e_plus_00 = [63.91693233306098, 63.91693278033812, 63.916932875895334, 63.91693292021378, 63.91693294534046, 63.916932961218684, 63.916932971956655, 63.91693297955131, 63.9169329850968, 63.916932989239385, 63.9169329923882, 63.91693299481267, 63.916932996698215]
frequency_invseconds_fluxx_for_all_energies_32_16_att_total_1e_min_12_att_matter_1e_plus_00_att_nu_nu_1e_plus_00_att_vacuum_1e_plus_00 = [6.391693233306097, 6.391693278033811, 6.3916932875895345, 6.391693292021379, 6.391693294534045, 6.391693296121868, 6.391693297195666, 6.391693297955133, 6.391693298509681, 6.39169329892394, 6.391693299238819, 6.391693299481267, 6.391693299669821]

frequency_invseconds_fluxx_all_att_32_16 = np.array([
    frequency_invseconds_fluxx_for_all_energies_32_16_att_total_1e_min_12_att_matter_1e_plus_00_att_nu_nu_1e_plus_00_att_vacuum_1e_plus_00,
    frequency_invseconds_fluxx_for_all_energies_32_16_att_total_1e_min_11_att_matter_1e_plus_00_att_nu_nu_1e_plus_00_att_vacuum_1e_plus_00,
    frequency_invseconds_fluxx_for_all_energies_32_16_att_total_1e_min_10_att_matter_1e_plus_00_att_nu_nu_1e_plus_00_att_vacuum_1e_plus_00,
    frequency_invseconds_fluxx_for_all_energies_32_16_att_total_1e_min_09_att_matter_1e_plus_00_att_nu_nu_1e_plus_00_att_vacuum_1e_plus_00,
    frequency_invseconds_fluxx_for_all_energies_32_16_att_total_1e_min_08_att_matter_1e_plus_00_att_nu_nu_1e_plus_00_att_vacuum_1e_plus_00,
    frequency_invseconds_fluxx_for_all_energies_32_16_att_total_1e_min_07_att_matter_1e_plus_00_att_nu_nu_1e_plus_00_att_vacuum_1e_plus_00,
    frequency_invseconds_fluxx_for_all_energies_32_16_att_total_1e_min_06_att_matter_1e_plus_00_att_nu_nu_1e_plus_00_att_vacuum_1e_plus_00,
    frequency_invseconds_fluxx_for_all_energies_32_16_att_total_1e_min_05_att_matter_1e_plus_00_att_nu_nu_1e_plus_00_att_vacuum_1e_plus_00,
])

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
for i in range(len(energybinsMeV_0_13)):
    ax.plot(
        np.log10(eta_for_energy_plots),
        np.log10(sin_2thetaeff_fluxx_all_att_32_16[:,i]),
        label=r'$\sin 2\theta_{\mathrm{eff}}$ for $E = %.3f$' % energybinsMeV_0_13[i],
        linewidth=3
    )
ax.plot(np.log10(eta), np.log10(rho_eu_32_16), label='Emu', marker='o', linewidth=3, markersize=12)
# ax.axhline(0, color='gray', linewidth=3, linestyle='--')
# ax.axvline(0, color='gray', linewidth=3, linestyle='--')
ax.set_xlabel(r'$\log \eta$')
ax.set_ylabel(r'$\log \rho_{eu}$')
ax.set_title('Quantum coherence in the disk \nwith self interaction hamiltonian along $\\hat{p}=\\hat{x}$')
leg = ax.legend(framealpha=0.0, ncol=2, fontsize=12)
pf.apply_custom_settings(ax, leg, log_scale_y=False)
plt.show()
plt.close(fig)

fig, ax = plt.subplots(figsize=(10, 6))
for i in range(len(energybinsMeV_0_13)):
    ax.plot(np.log10(eta_for_energy_plots), np.log10(frequency_invseconds_fluxx_all_att_32_16[:,i]/pc.PhysConst.c), label=f'$E = {energybinsMeV_0_13[i]}$', linewidth=3)
# ax.axhline(0, color='gray', linewidth=3, linestyle='--')
# ax.axvline(0, color='gray', linewidth=3, linestyle='--')
ax.set_xlabel(r'$\log \eta$')
ax.set_ylabel(r'$\log\,\omega\,[\,\mathrm{cm}^{-1}\,]$')
ax.set_title('Effective frequency in the disk \nwith self interaction hamiltonian along $\\hat{p}=\\hat{x}$')
leg = ax.legend(framealpha=0.0, ncol=1, fontsize=12)
pf.apply_custom_settings(ax, leg, log_scale_y=False)
plt.show()
plt.close(fig)

# Theoretical effective vacuum evolution

In [ ]:
def rho_effective_vacuum_theoretical(theta, omega, t):
    """
    Two-flavor neutrino density matrix in vacuum.

    Parameters
    ----------
    theta : float
        Vacuum mixing angle (radians)
    omega : float
        Oscillation frequency = Δm²/(2E)
    t : float
        Time

    Returns
    -------
    rho : complex ndarray (2,2)
    """
    
    s2 = np.sin(2.0 * theta)
    c2 = np.cos(2.0 * theta)

    x = 0.5 * omega * t

    Pee = 1.0 - s2**2 * np.sin(x)**2
    Pex = s2**2 * np.sin(x)**2

    rho_ex = (
        s2 * np.sin(x)
        * (-c2 * np.sin(x) + 1j * np.cos(x))
    )

    rho = np.array([
        [Pee, np.conj(rho_ex)],
        [rho_ex, Pex]
    ], dtype=complex)

    return rho

In [ ]:
Length = 1e5 #cm
Length_km = Length/1e5
Time = Length/pc.PhysConst.c #s

theta_eff_fluxx_all_att_32_16 = 0.5*np.arcsin(sin_2thetaeff_fluxx_all_att_32_16)
theta_eff_fluxz_all_att_48_48 = 0.5*np.arcsin(sin_2thetaeff_fluxz_all_att_48_48)

rho_theoretical_fluxx_all_att_32_16 = rho_effective_vacuum_theoretical(theta_eff_fluxx_all_att_32_16, frequency_invseconds_fluxx_all_att_32_16, Time)
rho_theoretical_fluxz_all_att_48_48 = rho_effective_vacuum_theoretical(theta_eff_fluxz_all_att_48_48, frequency_invseconds_fluxz_all_att_48_48, Time)

off_diagonal_rho_theoretical_fluxx_all_att_32_16 = np.abs(rho_theoretical_fluxx_all_att_32_16[0,1])
off_diagonal_rho_theoretical_fluxz_all_att_48_48 = np.abs(rho_theoretical_fluxz_all_att_48_48[0,1])

In [ ]:
rho_eu_equation_title = (
    r'$\rho_{eu}=\left|\sin(2\theta_{\rm eff})\sin\!\left(\frac{\omega_{\rm eff} t}{2}\right)'
    r'\left[-\cos(2\theta_{\rm eff})\sin\!\left(\frac{\omega_{\rm eff} t}{2}\right)'
    r'+i\cos\!\left(\frac{\omega_{\rm eff} t}{2}\right)\right]\right|$'
)

fig, ax = plt.subplots(figsize=(10, 6))
for i in range(len(energybinsMeV_0_13)):
    ax.plot(
        np.log10(eta_for_energy_plots),
        np.log10(off_diagonal_rho_theoretical_fluxx_all_att_32_16[:,i]),
        label=fr'$\rho_{{eu}}$ for $E = {energybinsMeV_0_13[i]:.3f}$ MeV',
   
        linewidth=3
    )
ax.plot(np.log10(eta), np.log10(rho_eu_32_16), label='Emu', marker='o', linewidth=3, markersize=12)
# ax.axhline(0, color='gray', linewidth=3, linestyle='--')
# ax.axvline(0, color='gray', linewidth=3, linestyle='--')
ax.set_xlabel(r'$\log \eta$')
ax.set_ylabel(r'$\log \rho_{eu}$')
ax.set_title(
    r'Quantum coherence in the disk '
    '\n'
    r'with self interaction hamiltonian along $\hat{p}=\hat{x}$'
    '\n'
    + rho_eu_equation_title + 
    '\n'
    r'$t = ' + str(Length_km) + r'\,{\rm km}/c$'
)
leg = ax.legend(framealpha=0.0, ncol=2, fontsize=12)
pf.apply_custom_settings(ax, leg, log_scale_y=False)
plt.show()
plt.close(fig)

fig, ax = plt.subplots(figsize=(10, 6))
for i in range(len(energybinsMeV_0_13)):
    ax.plot(
        np.log10(eta_for_energy_plots),
        np.log10(off_diagonal_rho_theoretical_fluxz_all_att_48_48[:,i]),
        label=fr'$\rho_{{eu}}$ for $E = {energybinsMeV_0_13[i]:.3f}$ MeV',
   
        linewidth=3
    )
ax.plot(np.log10(eta), np.log10(rho_eu_48_48), label='Emu', marker='o', linewidth=3, markersize=12)
# ax.axhline(0, color='gray', linewidth=3, linestyle='--')
# ax.axvline(0, color='gray', linewidth=3, linestyle='--')
ax.set_xlabel(r'$\log \eta$')
ax.set_ylabel(r'$\log \rho_{eu}$')
ax.set_title(
    r'Quantum coherence in the pole '
    '\n'
    r'with self interaction hamiltonian along $\hat{p}=\hat{z}$'
    '\n'
    + rho_eu_equation_title + 
    '\n'
    r'$t = ' + str(Length_km) + r'\,{\rm km}/c$'
)
leg = ax.legend(framealpha=0.0, ncol=2, fontsize=12)
pf.apply_custom_settings(ax, leg, log_scale_y=False)
plt.show()
plt.close(fig)